# Purpose

Prove the P2D specialist and regulated activation is bounded, deterministic, and complete.

## Lemma: L75

In [ ]:
from pathlib import Path
import yaml

repo_root = Path.cwd()

agents = yaml.safe_load((repo_root / 'registry' / 'agents.yaml').read_text(encoding='utf-8'))
persona_reg = yaml.safe_load((repo_root / 'registry' / 'persona_registry_v2.yaml').read_text(encoding='utf-8'))
cap_map = yaml.safe_load((repo_root / 'registry' / 'persona_capability_map.yaml').read_text(encoding='utf-8'))
routing = yaml.safe_load((repo_root / 'registry' / 'executor_routing_v2.yaml').read_text(encoding='utf-8'))

ACTIVATED = {
    'ai-engineer': (10, 'engineering'),
    'frontend-developer': (7, 'engineering'),
    'mobile-app-builder': (7, 'engineering'),
    'rapid-prototyper': (8, 'engineering'),
    'test-writer-fixer': (7, 'engineering'),
    'performance-benchmarker': (8, 'testing'),
    'test-results-analyzer': (7, 'testing'),
    'tool-evaluator': (7, 'testing'),
    'workflow-optimizer': (8, 'testing'),
    'finance-tracker': (8, 'studio-operations'),
    'infrastructure-maintainer': (8, 'studio-operations'),
    'legal-compliance-checker': (8, 'studio-operations'),
    'visual-storyteller': (7, 'design'),
    'whimsy-injector': (7, 'design'),
    'experiment-tracker': (8, 'project-management'),
    'trend-researcher': (7, 'product'),
}

BLOCKED = ['instagram-curator', 'tiktok-strategist', 'reddit-community-builder',
           'twitter-engager', 'app-store-optimizer']

prefix_routes = routing['prefix_routes']

for pid, (expected, dept) in ACTIVATED.items():
    p = persona_reg['personas'][pid]
    assert p['coverage_status'] == 'registry-backed', f'{pid} not registry-backed'
    assert p['status'] == 'active', f'{pid} not active'
    a = agents['agents'][pid]
    assert len(a['allowed_actions']) == expected, f'{pid}: {len(a["allowed_actions"])} != {expected}'
    c = cap_map['departments'][dept]['personas'][pid]
    assert c['coverage_status'] == 'registry-backed', f'{pid} cap not rb'
    assert c['current_action_count'] == expected, f'{pid} cap count mismatch'
    for action in a['allowed_actions']:
        prefix = action.split('.')[0] + '.'
        assert prefix in prefix_routes, f'{pid} action {action} unroutable'

total = sum(n for n, _ in ACTIVATED.values())
assert total == 122, f'Expected 122 actions, got {total}'

for pid in BLOCKED:
    p = persona_reg['personas'][pid]
    assert p['coverage_status'] == 'persona-contract-only', f'{pid} should be contract-only'
    a = agents['agents'][pid]
    assert len(a['allowed_actions']) == 0, f'{pid} should have 0 actions'

for pid in ['finance-tracker', 'legal-compliance-checker']:
    a = agents['agents'][pid]
    assert 'approval_rules' in a, f'{pid} missing approval_rules'

print('ALL L75 CHECKS PASSED')